In [29]:
import pandas as pd 
import numpy as np
from datetime import datetime, date
import os 

In [30]:
from aind_data_access_api.document_db import MetadataDbClient

API_GATEWAY_HOST = "api.allenneuraldynamics.org"
DATABASE = 'metadata_index'
COLLECTION = 'data_assets'

docdb_api_client = MetadataDbClient(
   host=API_GATEWAY_HOST,
   database=DATABASE,
   collection=COLLECTION,
)
print(docdb_api_client._base_url)

https://api.allenneuraldynamics.org/v1/metadata_index/data_assets


In [40]:
aggregate = [
  {
    "$match": {
      "data_description.project_name": {"$in": ["Cell Type Lookup Table", "Cell Type LUT"]}, 
      "name": {"$regex": "nwb"},
      "location": {"$regex": "aind-open-data"}, 
      "data_description": {"$ne": None}
    }
  },
  {
    "$project": {
      "name": 1, 
      "subject_id": "$data_description.subject_id",
      "genotype": "$subject.subject_details.genotype", 
      "date_of_birth": "$subject.subject_details.date_of_birth", 
      "sex": "$subject.subject_details.sex",  
      "session_start_time": "$acquisition.acquisition_start_time",
      "session_end_time": "$acquisition.acquisition_end_time", 
      "stimulus_epochs": "$acquisition.stimulus_epochs.stimulus_name", 
      "project_name": "$data_description.project_name", 
      "modality": "$data_description.modalities.name"
    }
  },
]
    
records = docdb_api_client.aggregate_docdb_records(
    pipeline=aggregate,
)

In [53]:
df = pd.DataFrame(records)
df.head()

,_id,name,subject_id,genotype,date_of_birth,sex,session_start_time,session_end_time,stimulus_epochs,project_name,modality
0,be7e6c7f-c5a2-420e-8b0d-f709353df6f1,ecephys_655565_2023-04-05_16-17-25_nwb_2025-06...,655565,Chat-IRES-Cre-neo/Chat-IRES-Cre-neo,2022-10-24,Male,2023-04-05T16:17:25-07:00,2023-04-05T17:05:43-07:00,[Laser pulses],Cell Type LUT,"[Extracellular electrophysiology, Behavior vid..."
1,f8e4c6d5-7176-4f21-9b88-d04238fda04c,ecephys_655568_2023-05-01_15-26-47_nwb_2025-06...,655568,Chat-IRES-Cre-neo/Chat-IRES-Cre-neo,2022-10-24,Female,2023-05-01T15:26:47-07:00,2023-05-01T16:13:33-07:00,[Laser pulses],Cell Type LUT,"[Extracellular electrophysiology, Behavior vid..."
2,5826720c-6582-4ea7-94e0-4c9aa4afa7a9,ecephys_655568_2023-05-03_15-21-12_nwb_2025-06...,655568,Chat-IRES-Cre-neo/Chat-IRES-Cre-neo,2022-10-24,Female,2023-05-03T15:21:12-07:00,2023-05-03T16:07:54-07:00,[Laser pulses],Cell Type LUT,"[Extracellular electrophysiology, Behavior vid..."
3,01ecc549-edaf-4594-885d-a12f97e87599,ecephys_655571_2023-05-15_13-39-49_nwb_2025-06...,655571,Chat-IRES-Cre-neo/Chat-IRES-Cre-neo,2022-10-24,Female,2023-05-15T13:39:49-07:00,2023-05-15T14:26:36-07:00,[Laser pulses],Cell Type LUT,"[Extracellular electrophysiology, Behavior vid..."
4,714a99ce-6734-415c-8fa5-287d33a2fd27,ecephys_655572_2023-05-09_15-03-29_nwb_2025-06...,655572,Chat-IRES-Cre-neo/Chat-IRES-Cre-neo,2022-10-24,Female,2023-05-09T15:03:29-07:00,2023-05-09T15:51:02-07:00,[Laser pulses],Cell Type LUT,"[Extracellular electrophysiology, Behavior vid..."


In [ ]:
to_exclude = ['ecephys_655565_2023-03-31_14-47-36_nwb_2025-07-16_16-52-27'] 

filtered_df = df[~df.name.isin(to_exclude)]

40

In [56]:
filtered_df.to_csv('/data/metadata/cell_type_look_up_table_metadata.csv', index = False)